# A Loan, Not a Currency

### The stochastic break test: what noise can and cannot buy a loop

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The question, stated so it can be answered wrong

The previous notebook reduced memory and counting to one phase-closure condition on a cyclic loop,

$$N\arccos(1/G) + \tau\sqrt{G^2-1} = \Phi$$

and named the untested surface: **finite copy number.** Everything so far was deterministic — a mean-field
limit with no fluctuation. A real substrate has discrete molecules, and below the deterministic threshold a
noisy loop *does* oscillate. That effect is real, measurable, and known as coherence resonance.

So the framework faces a fork it cannot dodge:

> **Is noise a fifth currency** — a resource a substrate can spend to buy a capability it could not otherwise
> afford — **or is it a loan** that appears as oscillation at small scale and disappears as the substrate grows?

Those two possibilities make opposite quantitative predictions, which is what makes this a test rather than a
framing exercise:

| | if noise is a **currency** | if noise is a **loan** |
|---|---|---|
| relative amplitude as $\Omega\to\infty$ | survives, $\mathrm{CV}\to$ const | vanishes as $\Omega^{-1/2}$ |
| coherence time | grows with $\Omega$, like a real clock | pinned at the deterministic damping time $1/\lvert\mathrm{Re}\,\lambda\rvert$ |
| frequency | something new, set by the noise | the deterministic resonance $\mathrm{Im}\,\lambda$ |
| threshold | smeared away | still locatable, and at $\sec(\pi/N)$ |

**Test substrate.** A 3-ring of mutual repressors simulated as a discrete chemical master equation by
tau-leaping, with system size $\Omega$ setting molecule counts. Its deterministic threshold is
$G>\sec(\pi/3)=2$, and the least-stable circulant eigenvalue pair is

$$\lambda_{1,2} = -1 - G\,e^{\pm 2\pi i/3} = \left(\tfrac{G}{2}-1\right) \pm i\,\frac{G\sqrt3}{2}$$

so the deterministic law hands over two numbers to check the noisy regime against, fixed before any
simulation runs: a damping rate $\lvert G/2-1\rvert$ and a resonant frequency $G\sqrt3/2$.

---

## Step 1 — Library


In [1]:
import numpy as np
from scipy.optimize import brentq

# ===========================================================================
# Deterministic reference: dx_i/dt = a/(1+x_{i-1}^h) - x_i  on a 3-ring.
# Symmetric state x* = a/(1+x*^h);   G = |f'(x*)| = h x*^h/(1+x*^h)
# Circulant eigenvalues lam_k = -1 - G e^{2 pi i k/3}
#   least-stable pair:  Re = G/2 - 1      (unstable iff G > 2 = sec(pi/3))
#                       Im = G sqrt(3)/2  (the resonant frequency)
# ===========================================================================
def xstar(a, h):
    return brentq(lambda x: a / (1 + x**h) - x, 1e-14, a, xtol=1e-15)

def gain(a, h):
    x = xstar(a, h)
    return h * x**h / (1 + x**h)

def design_for_gain(G, ratio=1.25):
    """Closed form: pick a target per-stage gain G, get (a, h, x*). Needs h > G."""
    h = G * ratio
    un = G / (h - G)
    u = un ** (1.0 / h)
    return u * h / (h - G), h, u

def eig_pair(G):
    """(damping rate Re lam, resonant frequency Im lam) of the least-stable pair."""
    return G / 2 - 1.0, G * np.sqrt(3) / 2.0


# ===========================================================================
# Stochastic substrate: chemical master equation by tau-leaping.
#   production of X_i:   propensity  Omega * a / (1 + (n_{i-1}/Omega)^h)
#   degradation of X_i:  propensity  n_i
# Omega is the system size. Omega -> infinity recovers the ODE exactly.
# ===========================================================================
def tauleap(a, h, Om, T=1600.0, dt=2e-3, stride=25, reps=10, seed=0, burn=0.15):
    rng = np.random.default_rng(seed)
    x0 = xstar(a, h)
    n = np.maximum(np.rint(Om * x0 * np.ones((reps, 3)))
                   + rng.normal(0, np.sqrt(Om * x0), (reps, 3)), 0.0)
    steps = int(T / dt); kb = int(burn * steps); rec = []
    for k in range(steps):
        x = n / Om
        births = rng.poisson(a * Om * dt / (1.0 + np.roll(x, 1, axis=1) ** h))
        deaths = rng.poisson(n * dt)
        n = np.maximum(n + births - deaths, 0.0)
        if k >= kb and (k - kb) % stride == 0:
            rec.append(n / Om)
    return np.array(rec), dt * stride          # concentrations, sample interval


# ===========================================================================
# Measurements.
# The circulant order parameter  z = sum_k x_k e^{2 pi i k/3}  isolates the
# oscillatory mode. Its autocorrelation C(t) carries both answers at once:
#   |C(t)| ~ e^{-Gamma t}   Gamma = damping rate (below) or phase diffusion (above)
#   arg C(t) ~ omega t      the oscillation frequency
# Computed by FFT so the lag sweep costs N log N rather than N per lag.
# ===========================================================================
def order_parameter(traj):
    w = np.exp(2j * np.pi * np.arange(3) / 3)
    z = (traj * w).sum(axis=2)
    return z - z.mean(axis=0)

def decorrelation(traj, dts, maxlag=40.0, floor=0.08):
    z = order_parameter(traj)
    N = len(z); nf = 1 << int(np.ceil(np.log2(2 * N)))
    Z = np.fft.fft(z, n=nf, axis=0)
    ac = np.fft.ifft(Z * np.conj(Z), axis=0)[:N]
    Cf = (ac / np.arange(N, 0, -1)[:, None]).mean(axis=1)
    L = min(int(maxlag / dts), N)
    Cf = Cf[:L]; A = np.abs(Cf) / np.abs(Cf[0]); t = np.arange(L) * dts
    m = (A > floor) & (t > 0)
    if m.sum() < 8:
        return np.nan, np.nan
    Gamma = -np.polyfit(t[m], np.log(A[m]), 1)[0]
    omega = abs(np.polyfit(t[m], np.unwrap(np.angle(Cf[m])), 1)[0])
    return Gamma, omega

def cv(traj):
    """Relative amplitude: coefficient of variation of one species."""
    x = traj[:, :, 0]
    return float(np.mean(np.std(x, axis=0) / np.maximum(np.mean(x, axis=0), 1e-12)))

print("library ready")


library ready


---
## Step 2 — One pass, three measurements, both sides of the threshold

Two gains: $G=1.7$ (below the threshold of 2) and $G=2.3$ (above). Six system sizes each, spanning a factor
of 333 in molecule count. For every run: relative amplitude, decorrelation rate, and frequency — each against
a number the deterministic law fixed in advance.


In [2]:
print("3-ring CME. Deterministic threshold G > sec(pi/3) = 2.")
print("Predicted columns come from the deterministic eigenvalue, computed before simulating.\n")
print(f"{'G':>5} {'Omega':>6} | {'CV':>9} | {'Gamma':>8} {'|Re lam|':>9} | {'omega':>8} {'Im lam':>8}")
print("-" * 70)
R = {}
Oms = [30, 100, 300, 1000, 3000, 10000]
for Gt in [1.7, 2.3]:
    a, h, _ = design_for_gain(Gt)
    G = gain(a, h); re, im = eig_pair(G)
    for Om in Oms:
        tr, dts = tauleap(a, h, Om, seed=5)
        Gam, om = decorrelation(tr, dts); c = cv(tr)
        R[(Gt, Om)] = (c, Gam, om)
        print(f"{G:>5.2f} {Om:>6} | {c:>9.4f} | {Gam:>8.4f} {abs(re):>9.4f} | {om:>8.4f} {im:>8.4f}")
    print()

print("Scaling with system size (log-log slopes):")
o = np.array(Oms, float)
for Gt in [1.7, 2.3]:
    a, h, _ = design_for_gain(Gt); G = gain(a, h)
    c = np.array([R[(Gt, k)][0] for k in Oms])
    g = np.array([R[(Gt, k)][1] for k in Oms])
    print(f"   G={G:.2f}:   CV slope = {np.polyfit(np.log(o), np.log(c), 1)[0]:+.3f}"
          f"      Gamma slope = {np.polyfit(np.log(o), np.log(g), 1)[0]:+.3f}")
print("""
   CV slope     -1/2 -> amplitude is borrowed from fluctuation and vanishes with scale
                   0 -> amplitude is owned by the dynamics
   Gamma slope     0 -> coherence pinned at the deterministic damping rate
                  -1 -> phase diffusion ~ 1/Omega: coherence grows without bound
""")


3-ring CME. Deterministic threshold G > sec(pi/3) = 2.
Predicted columns come from the deterministic eigenvalue, computed before simulating.

    G  Omega |        CV |    Gamma  |Re lam| |    omega   Im lam
----------------------------------------------------------------------
 1.70     30 |    0.2715 |   0.1809    0.1500 |   1.3450   1.4722
 1.70    100 |    0.1557 |   0.1394    0.1500 |   1.4452   1.4722
 1.70    300 |    0.0904 |   0.1462    0.1500 |   1.4314   1.4722
 1.70   1000 |    0.0483 |   0.1581    0.1500 |   1.4742   1.4722
 1.70   3000 |    0.0282 |   0.1461    0.1500 |   1.4644   1.4722
 1.70  10000 |    0.0154 |   0.1630    0.1500 |   1.4748   1.4722

 2.30     30 |    0.5698 |   0.0886    0.1500 |   1.3663   1.9919
 2.30    100 |    0.5275 |   0.0523    0.1500 |   1.4229   1.9919
 2.30    300 |    0.5183 |   0.0195    0.1500 |   1.4361   1.9919
 2.30   1000 |    0.5154 |   0.0053    0.1500 |   1.4368   1.9919
 2.30   3000 |    0.5141 |   0.0018    0.1500 |   1.4366   1

Both sides answer cleanly, and the answers are opposite on every measure.

**Below threshold** ($G=1.70$): relative amplitude falls with slope $-0.497$ — textbook $\Omega^{-1/2}$. The
decorrelation rate is flat in $\Omega$ (slope $-0.008$) and sits near $0.15$, which is $\lvert G/2-1\rvert =
0.150$: the deterministic damping rate. The frequency converges to $1.475$ against a predicted
$\mathrm{Im}\,\lambda = 1.4722$.

So the sub-threshold oscillation is not a new phenomenon the law fails to cover. It is **the deterministic
eigenvalue, rung by noise** — frequency from $\mathrm{Im}\,\lambda$, decay from $\mathrm{Re}\,\lambda$,
amplitude from the fluctuation scale alone. The law describes it completely; it simply is not producing a
self-sustaining solution there. (At $\Omega=30$ the rate reads $0.181$ against $0.150$ — the finite-size
correction, which shrinks as $\Omega$ grows.)

**Above threshold** ($G=2.30$): amplitude is flat in $\Omega$ (slope $-0.015$), and the decorrelation rate
falls with slope $-0.893$, close to the $1/\Omega$ that phase diffusion predicts. Coherence time grows
without bound as the substrate grows. The measured frequency ($1.437$) does *not* match
$\mathrm{Im}\,\lambda=1.992$, and should not: that is the onset frequency, and a mature limit cycle is shifted
by its own amplitude. The linear law predicts where oscillation *begins*, not the frequency it settles into.

---
## Step 3 — Reading the threshold off the data

The strongest form of the test: do not tell the analysis where the threshold is. Sweep $G$ and measure the
decorrelation rate at two system sizes differing by a factor of 30.

Below onset the rate is set by deterministic damping, so both sizes must agree — ratio $\approx1$. Above onset
it is phase diffusion, so the larger system must stay coherent far longer — ratio $\gg1$. **Where the ratio
departs from 1 is the threshold**, located from simulation output alone.


In [3]:
print("Sweep G. Compare decorrelation rate at Omega=100 against Omega=3000.")
print("Nothing here is told where the threshold is.\n")
print(f"{'G':>6} {'|G/2-1|':>9} | {'Gamma(100)':>11} {'Gamma(3000)':>12} | {'ratio':>7}  verdict")
print("-" * 74)
for Gt in [1.4, 1.6, 1.8, 1.9, 2.0, 2.1, 2.2, 2.5]:
    a, h, _ = design_for_gain(Gt); G = gain(a, h)
    t1, d1 = tauleap(a, h, 100,  T=800.0, dt=4e-3, stride=12, seed=7)
    t2, d2 = tauleap(a, h, 3000, T=800.0, dt=4e-3, stride=12, seed=7)
    r1, _ = decorrelation(t1, d1); r2, _ = decorrelation(t2, d2)
    ratio = r1 / max(r2, 1e-9)
    verdict = "damped ring" if ratio < 1.35 else ("crossover" if ratio < 3 else "SELF-SUSTAINED")
    print(f"{G:>6.3f} {abs(G/2-1):>9.4f} | {r1:>11.4f} {r2:>12.4f} | {ratio:>7.2f}  {verdict}")
print("\n   sec(pi/3) = 2.000 -- and that is where the two system sizes part company.")


Sweep G. Compare decorrelation rate at Omega=100 against Omega=3000.
Nothing here is told where the threshold is.

     G   |G/2-1| |  Gamma(100)  Gamma(3000) |   ratio  verdict
--------------------------------------------------------------------------
 1.400    0.3000 |      0.3068       0.2736 |    1.12  damped ring
 1.600    0.2000 |      0.1937       0.1555 |    1.25  damped ring
 1.800    0.1000 |      0.1003       0.0977 |    1.03  damped ring
 1.900    0.0500 |      0.0658       0.0424 |    1.55  crossover
 2.000    0.0000 |      0.0618       0.0134 |    4.62  SELF-SUSTAINED
 2.100    0.0500 |      0.0573       0.0050 |   11.50  SELF-SUSTAINED
 2.200    0.1000 |      0.0631       0.0033 |   18.98  SELF-SUSTAINED
 2.500    0.2500 |      0.0282       0.0009 |   32.28  SELF-SUSTAINED

   sec(pi/3) = 2.000 -- and that is where the two system sizes part company.


The ratio stays near 1 through $G=1.4,1.6,1.8$, rises to $1.55$ at $1.9$, then runs away:
$4.6,\,11.5,\,19.0,\,32.3$ from $G=2.0$ up. And the measured sub-threshold rates track $\lvert G/2-1\rvert$
directly — $0.307$ vs $0.300$, $0.100$ vs $0.100$.

**The threshold is recovered blind between $G=1.9$ and $2.0$, and $\sec(\pi/3)=2$ exactly.** The early
departure at $1.9$ is expected: there the damping rate is only $0.05$, so the crossover has a finite width
that narrows as $\Omega$ grows. This run is shorter than Step 2's, so the sub-threshold ratios scatter to
$1.1$–$1.25$ rather than sitting exactly at $1$; the separation above threshold is an order of magnitude
larger than that scatter.

---
## Step 4 — The verdict

**Noise is a loan, not a currency.** Two independent reasons, each measured:

1. **What it buys shrinks as the substrate grows.** Amplitude scales as $\Omega^{-1/2}$. A currency a substrate
   holds does not become worthless when the substrate gets larger; this does. In the deterministic limit the
   purchase is gone entirely.
2. **It buys no coherence at all.** The sharper reason. Below threshold the coherence time is pinned at
   $1/\lvert\mathrm{Re}\,\lambda\rvert$ *regardless of how large the system is* — rate slope $-0.008$ across a
   333-fold range. Fluctuation can start the loop ringing; it cannot stop the ring from decaying. Only paying
   the actual bill buys that, and then coherence grows $\propto\Omega$.

In the framework's terms: noise supplies **amplitude on credit, repaid at the damping rate.** The repayment
schedule is $\mathrm{Re}\,\lambda$, set by the same characteristic equation as the bill itself. There is no
free continuation.

And the result that matters most for the claim under test: **the law does not degrade below threshold.** It
was reasonable to expect the deterministic condition to become an approximation once molecules were counted —
to smear, shift, or acquire a correction term. It does none of those. It predicts the sub-threshold frequency
($1.475$ vs $1.4722$), the sub-threshold decay rate ($\approx0.15$ vs $0.150$), and the location of the
transition ($\approx2.0$ vs exactly $2$). The threshold is not where the law stops applying. It is where the
loop stops needing the loan.

---
## Step 5 — A correction to the previous notebook's framing

The last notebook called $\Phi$ a *conserved bill* with *four interchangeable currencies*. That was too loose
in two places, and the stochastic result makes the looseness matter.

**First, $\Phi$ is a phase-closure requirement, not a conserved quantity.** Nothing conserves it in the
dynamical sense; it is a condition fixed by the loop's sign that any realization must satisfy in order to
cross into oscillation. "Bill" is the right intuition. "Conserved" was not.

**Second, the resources are not peers.** Only two genuinely interchange:

| Resource | What it is | Interchangeable? |
|---|---|---|
| Stages $N$ | pays $\arccos(1/G)$ of phase each | **yes** — with delay, at $\arccos(1/G)/\sqrt{G^2-1}$ |
| Delay $\tau$ | pays $\tau\sqrt{G^2-1}$ of phase | **yes** — the same rate, $0.6046$ per stage at $G=2$ |
| Gain $G$ | satisfies the magnitude condition; sets what a stage is worth | no — it *prices* the phase currencies rather than substituting for them |
| Mass $M$ | feasibility: whether separable maintained basins can exist at all | no — a separate constraint, not phase |
| Noise | amplitude on credit | **no** — a loan, repaid at $\mathrm{Re}\,\lambda$ |

So the disciplined statement is:

$$\boxed{\;\substack{\text{One phase-closure law governs the latch and clock branches. Several resources decide whether}\\ \text{a substrate can pay for a branch, and only stages and delay genuinely substitute.}}\;}$$

That is weaker than "one bill, four currencies," and it is what the computations support.

---
## Step 6 — What this settles, and what is still open

The thread began with `Sun()` and `Sun(a,b,c,d)` — the claim that learning exposes arguments already callable
rather than adding values to a fixed object. Across three notebooks that claim has become checkable, and the
checking has now survived four substrate changes: chemistry, silicon, delay, and discreteness.

The signature, with each argument's role identified rather than merely listed:

```text
Clock(
    sign_product,        # selects which instability branch is reachable at all
    stages, delay,       # pay phase closure; substitute for each other at a known rate
    gain, decay,         # satisfy the magnitude condition; price the phase currencies
    mass,                # feasibility of separable maintained basins
    noise_scale,         # borrows amplitude; buys no coherence
    initial_condition    # selects a basin when several attractors coexist
)
```

The arguments were always callable. What learning found was not more of them, but **which substitute for
which, and at what rate** — and here, that one of the apparent arguments is not a resource at all.

**Still open, ordered by what a failure would cost:**

- **Second-order stages.** Every phase count assumed first-order stages, each capped at $\pi/2$. A stage with
  inertia supplies up to $\pi$ alone, so the dimension bound should move — a *single* second-order stage with a
  negative loop should oscillate. The phase equation makes that prediction sharply and it has not been run.
- **Spatial extent.** Transport supplies phase the way delay does, but with dispersion. Whether a
  reaction–diffusion ring obeys the same closure condition with $\tau$ replaced by a transport time is untested.
- **Non-Hill transfer shapes.** Everything used $a/(1+x^n)$. The law only ever referenced $f'$ at the fixed
  point, so it should be shape-blind — but "should be" is what break tests are for.
- **The Copy verb.** Still the one place the verb set is known to change with depth: classical fan-out is free
  because many carriers share the signal, and no-cloning forbids it for a single quantum. Whether any other verb
  has a boundary like that is the open question from the first notebook, and nothing here touched it.
